In [ ]:
!pip -q install xgboost lightgbm catboost imbalanced-learn shap optuna skl2onnx onnx onnxruntime joblib tldextract ucimlrepo --upgrade

In [ ]:
import os, re, json, time, math, random, warnings, zipfile
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import urlparse
import tldextract

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, matthews_corrcoef,
                              balanced_accuracy_score, confusion_matrix, classification_report,
                              roc_curve, precision_recall_curve)

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE

import joblib

SEED = 42
random.seed(SEED); np.random.seed(SEED)

FIG_DIR = "artifacts/figures"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

sns.set_theme(style="whitegrid")
PALETTE = sns.color_palette("viridis", 10)

def savefig(name):
    """Save the current matplotlib figure into artifacts/figures and show it."""
    path = os.path.join(FIG_DIR, f"{name}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print("saved:", path)
    plt.show()

print("Environment ready.")


In [ ]:
URL_A = "https://raw.githubusercontent.com/GregaVrbancic/Phishing-Dataset/master/dataset_full.csv"
df_a_raw = pd.read_csv(URL_A)
TARGET = "phishing"
print("Dataset A:", df_a_raw.shape)
df_a_raw.head()


In [ ]:
# ---- Shared lexical/domain feature extractor (mirrors Dataset A's naming/semantics) ----
SUSPICIOUS_CHARS = ['.', '-', '_', '/', '?', '=', '@', '&', '!', ' ', '~', ',', '+', '*', '#', '$', '%']
CHAR_NAMES = ['dot','hyphen','underline','slash','questionmark','equal','at','and',
              'exclamation','space','tilde','comma','plus','asterisk','hashtag','dollar','percent']

def extract_shared_features(raw_url: str) -> dict:
    """Compute a feature row using the SAME column names/semantics as columns in
    Dataset A's dataset_full.csv, but purely from the URL string (no DNS/WHOIS),
    so it can be applied to any raw URL list (Dataset B, future live URLs, etc.)"""
    url = str(raw_url).strip()
    if not re.match(r'^[a-zA-Z]+://', url):
        url_full = 'http://' + url
    else:
        url_full = url

    parsed = urlparse(url_full)
    ext = tldextract.extract(url_full)
    domain = ext.domain + ('.' + ext.suffix if ext.suffix else '')

    feats = {}
    for ch, name in zip(SUSPICIOUS_CHARS, CHAR_NAMES):
        feats[f"qty_{name}_url"] = url.count(ch)
    feats["qty_tld_url"] = 1 if ext.suffix else 0
    feats["length_url"] = len(url)

    for ch, name in zip(SUSPICIOUS_CHARS, CHAR_NAMES):
        feats[f"qty_{name}_domain"] = domain.count(ch)
    feats["qty_vowels_domain"] = sum(domain.lower().count(v) for v in "aeiou")
    feats["domain_length"] = len(domain)
    feats["domain_in_ip"] = int(bool(re.match(r'^\d{1,3}(\.\d{1,3}){3}$', ext.domain))) if ext.domain else 0
    feats["server_client_domain"] = int(("server" in domain.lower()) or ("client" in domain.lower()))
    return feats

print(extract_shared_features("paypal-login-secure-update.verify-account.com/login?cmd=reset"))
